# Chapter 11: Neural Networks That Understand Language
## King - Man + Woman == ?

**Book:** Grokking Deep Learning  
**Author:** Andrew W. Trask  
**Publisher:** Manning Publications

## Learning Objectives

1. Understand the basic concepts of Natural Language Processing (NLP).
2. Learn how to represent words as numerical data.
3. Understand the Bag of Words representation.
4. Build a neural network to predict movie review sentiment.
5. Understand how an embedding layer works.
6. Explore how neural networks learn relationships between words.
7. Understand word analogies using learned word embeddings.

## Chapter Contents

1. What Does It Mean to Understand Language?
2. Natural Language Processing (NLP)
3. Supervised NLP
4. IMDB Movie Reviews Dataset
5. Capturing Word Correlation in Input Data
6. Predicting Movie Reviews
7. Introduction to an Embedding Layer
8. Interpreting the Output
9. Neural Architecture
10. Comparing Word Embeddings
11. What Is the Meaning of a Neuron?
12. Filling in the Blank
13. Meaning Is Derived from Loss
14. King - Man + Woman ~= Queen
15. Word Analogies
16. Summary

In [3]:
import numpy as np
import time
import sys

## 1. Natural Language Processing (NLP)

Natural Language Processing (NLP) is a field that focuses on enabling computers to process and understand human language automatically.

In this chapter, neural networks are applied to text data rather than only numerical or image data.

Common NLP tasks include:

- Identifying where words and sentences begin and end.
- Predicting the part of speech of a word.
- Recognizing named entities such as people and places.
- Predicting the sentiment of a sentence.
- Filling in missing information based on context.

Neural networks require numerical inputs. Therefore, text must first be converted into numerical representations before it can be processed by a neural network.

## 2. Capturing Word Correlation in Input Data

The Bag of Words approach represents a sentence by recording which words appear in it.

Each word in the vocabulary is assigned a position in a vector. A value of 1 indicates that a word appears in the sentence, while 0 indicates that it does not.

This representation allows a neural network to process text as numerical input.

In this example, the vocabulary contains four words: cat, the, dog, and sat.

In [4]:
onehots = {}

onehots['cat'] = np.array([1, 0, 0, 0])
onehots['the'] = np.array([0, 1, 0, 0])
onehots['dog'] = np.array([0, 0, 1, 0])
onehots['sat'] = np.array([0, 0, 0, 1])

sentence = ['the', 'cat', 'sat']

x = (
    onehots[sentence[0]]
    + onehots[sentence[1]]
    + onehots[sentence[2]]
)

print("Sent Encoding:", x)

Sent Encoding: [1 1 0 1]


### Explanation

The code creates a one-hot vector for each word in the vocabulary. The vectors for the words in the sentence are then added together to form a sentence representation.

The sentence is "the cat sat".

Expected output:

`Sent Encoding: [1 1 0 1]`

The output indicates that the words "cat", "the", and "sat" appear in the sentence, while "dog" does not.

This example follows the Bag of Words concept explained in Chapter 11, pages 192–193.

## 3. IMDB Movie Reviews Dataset

The book introduces a sentiment analysis task using movie reviews.

The dataset contains movie reviews paired with sentiment labels. The model learns to predict whether a review expresses a positive or negative sentiment.

The input consists of review text, while the target consists of sentiment labels.

Before training the neural network, the text must be processed into numerical representations that the model can understand.

In [7]:
import os

print("File yang tersedia:")
print(os.listdir('/content'))

File yang tersedia:
['.config', 'sample_data']


In [9]:
import urllib.request

base_url = "https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/"

urllib.request.urlretrieve(base_url + "reviews.txt", "/content/reviews.txt")
urllib.request.urlretrieve(base_url + "labels.txt", "/content/labels.txt")

print("Dataset berhasil diunduh!")

Dataset berhasil diunduh!


In [10]:
import os

with open('/content/reviews.txt', 'r', encoding='utf-8') as f:
    raw_reviews = f.readlines()

with open('/content/labels.txt', 'r', encoding='utf-8') as f:
    raw_labels = f.readlines()

print("Jumlah reviews:", len(raw_reviews))
print("Jumlah labels:", len(raw_labels))
print("\nContoh review:", raw_reviews[0][:200])
print("Contoh label:", raw_labels[0].strip())

Jumlah reviews: 25000
Jumlah labels: 25000

Contoh review: bromwell high is a cartoon comedy . it ran at the same time as some other programs about school life  such as  teachers  . my   years in the teaching profession lead me to believe that bromwell high  
Contoh label: positive


In [11]:
# Memisahkan setiap review menjadi kumpulan kata
tokens = list(map(lambda x: set(x.split(" ")), raw_reviews))

# Membuat vocabulary dari kata-kata unik
vocab = set()

for sent in tokens:
    for word in sent:
        if len(word) > 0:
            vocab.add(word)

vocab = list(vocab)

# Membuat pemetaan kata ke indeks
word2index = {}

for i, word in enumerate(vocab):
    word2index[word] = i

print("Jumlah vocabulary:", len(vocab))
print("Contoh kata:", vocab[:10])

Jumlah vocabulary: 74074
Contoh kata: ['halfway', 'itttttttt', 'delamere', 'dislodge', 'tyranosaurous', 'jobbo', 'silentbob', 'stamp', 'carping', 'eeda']


In [12]:
input_dataset = []

for sent in tokens:
    sent_indices = []

    for word in sent:
        if word in word2index:
            sent_indices.append(word2index[word])

    input_dataset.append(list(set(sent_indices)))

# Mengubah label menjadi angka
target_dataset = []

for label in raw_labels:
    if label.strip() == "positive":
        target_dataset.append(1)
    else:
        target_dataset.append(0)

print("Jumlah input:", len(input_dataset))
print("Jumlah target:", len(target_dataset))
print("Contoh input:", input_dataset[0][:10])
print("Contoh target:", target_dataset[0])

Jumlah input: 25000
Jumlah target: 25000
Contoh input: [44041, 9227, 34832, 58391, 56860, 5660, 20014, 57907, 36421, 48200]
Contoh target: 1
